# CSSE4011 Machine Learning and Edge AI Workshop

### 0. Connect to a runtime and change the runtime type to **"T4 GPU"**

Click the down-arrow button in the top-right corner, as shown in the screenshots below.

> **Important:** Please make sure you select **T4 GPU**, as using a different runtime may lead to much longer execution times.

<img src="https://github.com/YangLi309/CSSE4011-ML-Workshop/blob/main/screenshots/change_runtime_type.png?raw=true" width="300"> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<img src="https://github.com/YangLi309/CSSE4011-ML-Workshop/blob/main/screenshots/runtime_type.png?raw=true" width="300">

## 1. Install Required Packages and Upload MNIST Data

In [1]:
!git clone https://github.com/YangLi309/CSSE4011-ML-Workshop.git
%cd CSSE4011-ML-Workshop/

Cloning into 'CSSE4011-ML-Workshop'...
remote: Enumerating objects: 70047, done.
remote: Counting objects: 100% (70047/70047), done.
remote: Compressing objects: 100% (70045/70045), done.
remote: Total 70047 (delta 3), reused 70032 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (70047/70047), 19.22 MiB | 16.87 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Updating files: 100% (70004/70004), done.
/content/CSSE4011-ML-Workshop


In [14]:
# Install ONNX for model export
!pip3 install onnx onnxscript onnxruntime torchao


## 2. Introduction

### What is PyTorch?
PyTorch is an open-source deep learning framework known for its flexibility and ease of use, making it ideal for rapid prototyping and research.

### What is ONNX?
ONNX (Open Neural Network Exchange) is an open standard format for representing machine learning models, enabling interoperability between various frameworks and devices.

### Why are these important for Edge AI?
- **PyTorch**: Facilitates easy model development and training.
- **ONNX**: Allows exporting models to a standardized format for deployment across different platforms, including resource-constrained edge devices.

### What You Will Learn in This Workshop
- Load and adapt a pretrained model for a new task.
- Evaluate model performance.
- Prune the model to improve inference time.
- Export the model to ONNX format.

## 3. Load Pretrained AlexNet and Adapte the Model for MNIST 10

In [4]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import alexnet, AlexNet_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Current device:", device)

# Pretrained AlexNet with the final layer replaced for MNIST (10 classes)
model = alexnet(weights=AlexNet_Weights.DEFAULT)
model.classifier[6] = nn.Linear(4096, 10)  # For 10 classes
model = model.to(device)
print("Pretrained AlexNet model is loaded.")

Current device: cuda


## 4. Evaluate the Pretrained AlexNet on MNIST

In [5]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# Transform for MNIST images (which are 28x28 grayscale)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to AlexNet input size
    transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load MNIST test data
test_dataset = ImageFolder(root='./data/test', transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Evaluate the model (with modified classifier but before fine-tuning)
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy of AlexNet with modified classifier (before fine-tuning) on MNIST test data: {100 * correct / total:.2f}%")
print("Note: This accuracy is expected to be low as the model hasn't been finetuned for MNIST yet")

# Sample predictions
print("\nSample predictions:")
for i in range(min(10, len(all_preds))):
    print(f"True: {all_labels[i]}, Predicted: {all_preds[i]}")

Accuracy of AlexNet with modified classifier (before fine-tuning) on MNIST test data: 10.53%
Note: This accuracy is expected to be low as the model hasn't been finetuned for MNIST yet

Sample predictions:
True: 0, Predicted: 9
True: 0, Predicted: 9
True: 0, Predicted: 9
True: 0, Predicted: 9
True: 0, Predicted: 9
True: 0, Predicted: 9
True: 0, Predicted: 9
True: 0, Predicted: 9
True: 0, Predicted: 7
True: 0, Predicted: 2


## 5. Freeze All Layers Except Classifier

In [6]:
# Transfer learning: keep ImageNet features fixed and only train the new MNIST head.
for param in model.features.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

## 6. Load MNIST Train Dataset

In [7]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to AlexNet input size
    transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Assuming your dataset is stored in "./data/" with subfolders 0/, 1/, ..., 9/
train_dataset = ImageFolder(root='./data/train', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

## 7. Finetune the Classifier on MNIST

### Training epochs and corresponding model accuracy

- **1 epoch:** 96.45% accuracy  
- **5 epochs:** 97.02% accuracy  
- **10 epochs:** 99.30% accuracy  

> **Note:** The default setting is **1 epoch** for time efficiency, but you may increase the number of epochs to achieve better performance.

> **Reference:** Training for **1 epoch** should take approximately **3 minutes**. If it takes significantly longer, check whether your runtime type is set to **CPU** instead of **GPU**.

In [8]:
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# You can increase the number of epochs if you want to have higher accuracy
num_epochs = 1

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Wrap train_loader with tqdm for progress bar
    train_loader_tqdm = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')

    for images, labels in train_loader_tqdm:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Update progress bar
        train_loader_tqdm.set_postfix({
            'loss': running_loss/total,
            'acc': 100.*correct/total
        })

    # Print epoch statistics
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    print(f'\nEpoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%')

Epoch 1/1: 100%|██████████| 1875/1875 [03:03<00:00, 10.21it/s, loss=0.00359, acc=96.4]


Epoch 1/1 - Loss: 0.1148, Accuracy: 96.41%


## 8. Evaluate the Finetuned Model

In [9]:
# Evaluate the finetuned model on the test set (same loop structure as pre-training eval).
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy of AlexNet with modified classifier (after fine-tuning) on MNIST test data: {100 * correct / total:.2f}%")

# Sample predictions
print("\nSample predictions:")
for i in range(min(10, len(all_preds))):
    print(f"True: {all_labels[i]}, Predicted: {all_preds[i]}")

Accuracy of AlexNet with modified classifier (after fine-tuning) on MNIST test data: 98.14%

Sample predictions:
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0
True: 0, Predicted: 0


## 9. Apply Pruning (50% on Conv Layers)

In [10]:
import torch.nn as nn
from torch.nn.utils import prune
import copy

def prune_model(model):
    # Create a deep copy of the model to avoid modifying the original
    pruned_model = copy.deepcopy(model)

    # Apply pruning to the copied model
    # For AlexNet, we should focus on pruning the convolutional layers
    # The instruction mentioned 50% on Conv layers, so we'll focus on those
    for name, module in pruned_model.named_modules():
        if isinstance(module, nn.Conv2d):
            prune.l1_unstructured(module, name="weight", amount=0.5)
        # We can apply a lighter pruning to fully connected layers
        elif isinstance(module, nn.Linear):
            prune.l1_unstructured(module, name="weight", amount=0.3)

    # Make pruning permanent to ensure the weights are actually removed
    for name, module in pruned_model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)) and hasattr(module, 'weight_orig'):
            prune.remove(module, 'weight')

    return pruned_model

finetuned_model = model
pruned_model = prune_model(model)
print("Weight pruning completed successfully.")

In [18]:
import copy
import torch
from torchao.quantization import quantize_, Int8WeightOnlyConfig

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Current device:", device)

# Copy the trained model so the original finetuned_model is unchanged
quantized_model = copy.deepcopy(finetuned_model).to(device).eval()

# Create a dummy input matching your model input shape
dummy_input = torch.randn(1, 3, 224, 224, device=device)

# Run the original model once
with torch.no_grad():
    fp32_output = quantized_model(dummy_input)
print("FP32 model executed successfully.")
print("Output shape:", fp32_output.shape)

# Quantize the model in-place on GPU
quantize_(quantized_model, Int8WeightOnlyConfig())

# Compile the quantized model for better GPU performance
# quantized_model = torch.compile(quantized_model, mode="max-autotune")

# Run the quantized model
with torch.no_grad():
    int8_output = quantized_model(dummy_input)
print("INT8 quantized model executed successfully on GPU.")
print("Output shape:", int8_output.shape)

Current device: cuda
FP32 model executed successfully.
Output shape: torch.Size([1, 10])
INT8 quantized model executed successfully on GPU.
Output shape: torch.Size([1, 10])


In [19]:
import time
import os
import torch

# Evaluate a model and return accuracy and FPS
def evaluate_model(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    inference_times = []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            # Synchronize for accurate GPU timing
            if device.type == "cuda":
                torch.cuda.synchronize()
            start_time = time.time()

            outputs = model(images)

            if device.type == "cuda":
                torch.cuda.synchronize()
            end_time = time.time()

            inference_time = end_time - start_time
            inference_times.append(inference_time)

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    avg_inference_time = sum(inference_times) / len(inference_times)
    fps = 1.0 / (avg_inference_time / images.size(0))

    return accuracy, fps


# Save model and measure file size
def get_model_size_mb(model, file_name):
    torch.save(model.state_dict(), file_name)
    size_mb = os.path.getsize(file_name) / (1024 * 1024)
    os.remove(file_name)
    return size_mb


# Make sure both models are on the same device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
finetuned_model = finetuned_model.to(device).eval()
quantized_model = quantized_model.to(device).eval()

# Evaluate the finetuned model
print("Evaluating finetuned PyTorch model...")
finetuned_accuracy, finetuned_fps = evaluate_model(finetuned_model, test_loader, device)
finetuned_size = get_model_size_mb(finetuned_model, "finetuned_model.pth")

# Evaluate the quantized model
print("Evaluating quantized model...")
quantized_accuracy, quantized_fps = evaluate_model(quantized_model, test_loader, device)
quantized_size = get_model_size_mb(quantized_model, "quantized_model.pth")

# Print comparison table
print("\n----- Model Comparison -----")
print(f"{'Model':<20} {'Accuracy':<15} {'FPS':<12} {'Size (MB)':<12}")
print("-" * 60)
print(f"{'Finetuned':<20} {finetuned_accuracy:.2f}%{'':<10} {finetuned_fps:.2f}{'':<6} {finetuned_size:.2f}")
print(f"{'Quantized':<20} {quantized_accuracy:.2f}%{'':<10} {quantized_fps:.2f}{'':<6} {quantized_size:.2f}")
print("-" * 60)

# Calculate differences
acc_diff = quantized_accuracy - finetuned_accuracy
fps_diff = quantized_fps - finetuned_fps
fps_improvement = (fps_diff / finetuned_fps) * 100
size_diff = quantized_size - finetuned_size
size_reduction = ((finetuned_size - quantized_size) / finetuned_size) * 100

print(f"\nAccuracy change after quantization: {acc_diff:.2f}%")
print(f"FPS improvement after quantization: {fps_improvement:.2f}% ({fps_diff:.2f} FPS)")
print(f"Model size change after quantization: {size_diff:.2f} MB")
print(f"Model size reduction after quantization: {size_reduction:.2f}%")

# Note on performance improvements
print("\nNote: GPU-based quantization may not always lead to a large speedup for small models or small batch sizes.")
print("The actual performance gain depends on the model structure, input size, hardware, and compiler optimizations.")
print("For larger models and heavier workloads, quantization benefits are often more noticeable.")

Evaluating finetuned PyTorch model...
Evaluating quantized model...

----- Model Comparison -----
Model                Accuracy        FPS          Size (MB)   
------------------------------------------------------------
Finetuned            98.85%           1340.43       217.61
Quantized            98.85%           1151.25       61.59
------------------------------------------------------------

Accuracy change after quantization: 0.00%
FPS improvement after quantization: -14.11% (-189.18 FPS)
Model size change after quantization: -156.02 MB
Model size reduction after quantization: 71.70%

Note: GPU-based quantization may not always lead to a large speedup for small models or small batch sizes.
The actual performance gain depends on the model structure, input size, hardware, and compiler optimizations.
For larger models and heavier workloads, quantization benefits are often more noticeable.


## 10. Compare Finetuned Model and Pruned Model

In [13]:
import time
import os
import torch

# Evaluate both models and compare accuracy, performance, and model size
def evaluate_model(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    inference_times = []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            # Measure inference time
            start_time = time.time()
            outputs = model(images)
            end_time = time.time()

            inference_time = end_time - start_time
            inference_times.append(inference_time)

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    avg_inference_time = sum(inference_times) / len(inference_times)
    fps = 1.0 / (avg_inference_time / images.size(0))  # FPS = batch_size / avg_time_per_batch

    return accuracy, fps


def get_model_size_mb(model, file_name):
    torch.save(model.state_dict(), file_name)
    size_mb = os.path.getsize(file_name) / (1024 * 1024)
    os.remove(file_name)
    return size_mb


# Evaluate the finetuned model
print("Evaluating finetuned model...")
finetuned_accuracy, finetuned_fps = evaluate_model(finetuned_model, test_loader, device)
finetuned_size = get_model_size_mb(finetuned_model, "finetuned_model.pth")

# Evaluate the pruned model
print("Evaluating pruned model...")
pruned_accuracy, pruned_fps = evaluate_model(pruned_model, test_loader, device)
pruned_size = get_model_size_mb(pruned_model, "pruned_model.pth")

# Print comparison results
print("\n----- Model Comparison -----")
print(f"{'Model':<20} {'Accuracy':<15} {'FPS':<12} {'Size (MB)':<12}")
print("-" * 60)
print(f"{'Finetuned':<20} {finetuned_accuracy:.2f}%{'':<10} {finetuned_fps:.2f}{'':<6} {finetuned_size:.2f}")
print(f"{'Pruned (50%)':<20} {pruned_accuracy:.2f}%{'':<10} {pruned_fps:.2f}{'':<6} {pruned_size:.2f}")
print("-" * 60)

# Calculate differences
acc_diff = pruned_accuracy - finetuned_accuracy
fps_diff = pruned_fps - finetuned_fps
fps_improvement = (fps_diff / finetuned_fps) * 100
size_diff = pruned_size - finetuned_size
size_reduction = ((finetuned_size - pruned_size) / finetuned_size) * 100

print(f"\nAccuracy change after pruning: {acc_diff:.2f}%")
print(f"FPS improvement after pruning: {fps_improvement:.2f}% ({fps_diff:.2f} FPS)")
print(f"Model size change after pruning: {size_diff:.2f} MB")
print(f"Model size reduction after pruning: {size_reduction:.2f}%")

# Note on performance improvements
print("\nNote: The model inference speed improvement here may not be very obvious due to I/O limitations.")
print("When using larger models and higher-resolution images, the difference is likely to be more noticeable.")
print("Factors such as memory bandwidth, disk I/O, and data preprocessing can mask the actual computational benefits of pruning.")
print("In real-world deployment scenarios with larger models (e.g., ResNet or EfficientNet) and higher-resolution images,")
print("the performance gains from pruning are often more significant.")


import torch
import torch.nn as nn

def check_sparsity(model):
    total_params = 0
    zero_params = 0

    print("Layer-wise sparsity:")
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            weight = module.weight.detach()
            num_params = weight.numel()
            num_zeros = torch.sum(weight == 0).item()
            sparsity = 100.0 * num_zeros / num_params

            total_params += num_params
            zero_params += num_zeros

            print(f"{name:<20} zeros: {num_zeros:>10} / {num_params:<10} sparsity: {sparsity:.2f}%")

    overall_sparsity = 100.0 * zero_params / total_params
    print(f"\nOverall sparsity: {overall_sparsity:.2f}%")

print("Finetuned model")
check_sparsity(finetuned_model)

print("\nPruned model")
check_sparsity(pruned_model)

Evaluating finetuned model...
Evaluating pruned model...

----- Model Comparison -----
Model                Accuracy        FPS          Size (MB)   
------------------------------------------------------------
Finetuned            98.85%           12486.33       217.61
Pruned (50%)         90.24%           12544.75       217.61
------------------------------------------------------------

Accuracy change after pruning: -8.61%
FPS improvement after pruning: 0.47% (58.43 FPS)
Model size change after pruning: -0.00 MB
Model size reduction after pruning: 0.00%

Note: The model inference speed improvement here may not be very obvious due to I/O limitations.
When using larger models and higher-resolution images, the difference is likely to be more noticeable.
Factors such as memory bandwidth, disk I/O, and data preprocessing can mask the actual computational benefits of pruning.
In real-world deployment scenarios with larger models (e.g., ResNet or EfficientNet) and higher-resolution images

## 11. Export Finetuned Model and Pruned Model to ONNX

In [ ]:
# ONNX export needs an example batch with the correct shape (NCHW); random values are fine—only the graph is fixed.
dummy_input = torch.randn(1, 3, 224, 224).to(device)

# Classic torch.onnx exporter: writes a single .onnx file (opset 18 matches current torch.onnx defaults).
torch.onnx.export(finetuned_model, dummy_input, "../alexnet_mnist.onnx", opset_version=18)  # finetuned
torch.onnx.export(pruned_model, dummy_input, "../alexnet_mnist_pruned.onnx", opset_version=18)  # pruned

In [ ]:
# Optional: newer export path (torch.export + Dynamo). Run after the cell that defines dummy_input.
# eval() disables dropout etc. so the exported graph matches inference.
finetuned_model.eval()
onnx_program = torch.onnx.export(
    finetuned_model,
    (dummy_input,),
    dynamo=True,
)
onnx_program.save("../alexnet_mnist.onnx")

pruned_model.eval()
onnx_program_pruned = torch.onnx.export(
    pruned_model,
    (dummy_input,),
    dynamo=True,
)
onnx_program_pruned.save("../alexnet_mnist_pruned.onnx")

12. Verify that the exported ONNX models match their PyTorch counterparts (same inputs, same logits within tolerance).

In [ ]:
# Same random input for both checks: PyTorch logits should match ONNXRuntime for each exported model.
import numpy as np
import onnxruntime as ort


def verify_onnx(torch_model, onnx_path, x, label):
    torch_model.eval()
    with torch.no_grad():
        torch_out = torch_model(x).cpu().numpy()
    sess = ort.InferenceSession(onnx_path)
    in_name = sess.get_inputs()[0].name
    onnx_out = sess.run(None, {in_name: x.cpu().numpy()})[0]
    ok = np.allclose(torch_out, onnx_out, atol=1e-4)
    mad = float(np.max(np.abs(torch_out - onnx_out)))
    print(f"{label}: match={ok}, max abs diff={mad:.6e}")


device = next(finetuned_model.parameters()).device
dummy_input = torch.randn(1, 3, 224, 224, device=device)

verify_onnx(finetuned_model, "../alexnet_mnist.onnx", dummy_input, "Finetuned")
verify_onnx(pruned_model, "../alexnet_mnist_pruned.onnx", dummy_input, "Pruned")

## 12. Recap

In this workshop you adapted pretrained AlexNet to MNIST, fine-tuned the classifier, compared accuracy and throughput against a pruned variant, exported both networks to ONNX, and checked that ONNX Runtime matches PyTorch on the same random input.

**Exported models** (paths follow your export cells, e.g. `../` relative to this notebook):

- **`alexnet_mnist.onnx`** — fine-tuned weights; use as the default deployment artifact.
- **`alexnet_mnist_pruned.onnx`** — pruned weights for studying sparsity and speed–accuracy trade-offs.

You can run these in ONNX Runtime, TensorRT, OpenVINO, or any runtime that supports the ONNX opset you exported with.
